# First-contact emitter assignment with the emitter-aware ontology

This notebook ingests the first `PY_CONTACT_LOG` line from `LuaHistory_2026-06-23.txt`, extracts CMO observation features, retrieves platform/operator hypotheses from the new KG ontology, applies the probability layer, and builds an LLM explanation payload.

In [ ]:
from pathlib import Path
import os, sys, json, math
from dataclasses import asdict

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'combat_id_calibration').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

LUA_HISTORY = REPO_ROOT / 'LuaHistory_2026-06-23.txt'
WORK_DIR = REPO_ROOT / 'notebooks' / 'outputs' / 'first_contact_emitter_assignment'
WORK_DIR.mkdir(parents=True, exist_ok=True)

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or None

In [ ]:
from combat_id_calibration.cmo_observation_ingest import parse_observation_line, write_observations_jsonl, populate_observations_neo4j
first_line = next(line for line in LUA_HISTORY.read_text(encoding='utf-8-sig', errors='replace').splitlines() if line.startswith('PY_CONTACT_LOG'))
obs = parse_observation_line(first_line, source_line=1)
write_observations_jsonl([obs], WORK_DIR / 'first_contact_observation.jsonl')
asdict(obs)

In [ ]:
# Optional: write the dynamic observation into the same Neo4j graph.
if NEO4J_PASSWORD:
    populate_observations_neo4j([obs], NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
    print('First observation ingested into Neo4j')
else:
    print('Set NEO4J_PASSWORD to ingest the observation into Neo4j')

In [ ]:
from combat_id_calibration.hypothesis_generation import fetch_graph_hypotheses, select_offline_hypotheses, emitter_aliases

seed_candidates = [
    {'hypothesis':'MiG-29 Fulcrum C', 'operator_nation':'Ukraine', 'emitter_aliases':['Slot Back [N-010 Zhuk-M]','N-010 Zhuk-M','Zhuk-M'], 'platform_class':obs.emission_target_type, 'typical_speed_kt':[250,800], 'typical_altitude_m':[0,18000], 'kg_support_count':1, 'evidence_paths':['offline_seed']},
    {'hypothesis':'MiG-29SMT', 'operator_nation':'Russia', 'emitter_aliases':['N-010 Zhuk-M','Zhuk-M'], 'platform_class':obs.emission_target_type, 'typical_speed_kt':[250,800], 'typical_altitude_m':[0,18000], 'kg_support_count':1, 'evidence_paths':['offline_seed']},
    {'hypothesis':'MiG-35', 'operator_nation':'Russia', 'emitter_aliases':['Zhuk-M','Zhuk-AE'], 'platform_class':obs.emission_target_type, 'typical_speed_kt':[250,900], 'typical_altitude_m':[0,17500], 'kg_support_count':1, 'evidence_paths':['offline_seed']},
]
if NEO4J_PASSWORD:
    kg_rows = fetch_graph_hypotheses(obs, 10, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE) or seed_candidates
else:
    kg_rows = seed_candidates
hypotheses = select_offline_hypotheses(obs, kg_rows, 5)
hypotheses

In [ ]:
def compatibility(obs, h):
    alias = any(a.lower() in obs.emission_sensor_name.lower() for a in h.get('emitter_aliases', []))
    speed_low, speed_high = h.get('typical_speed_kt', [0, 2500])
    alt_low, alt_high = h.get('typical_altitude_m', [0, 25000])
    score = 0.15 + 0.35*alias + 0.20*(speed_low <= obs.emission_speed <= speed_high) + 0.20*(alt_low <= obs.emission_altitude <= alt_high) + 0.10*min(float(h.get('kg_support_count',0)),5)/5
    return score

raw = [compatibility(obs, h) for h in hypotheses]
total = sum(raw) or 1
assignments = [{**h, 'probability': r/total, 'features': {'emitter_alias_match': any(a.lower() in obs.emission_sensor_name.lower() for a in h.get('emitter_aliases', [])), 'observed_speed_kt': obs.emission_speed, 'observed_altitude_m': obs.emission_altitude, 'observed_latitude': obs.emission_latitude, 'observed_longitude': obs.emission_longitude}} for h, r in zip(hypotheses, raw)]
(WORK_DIR / 'first_contact_probability_assignment.jsonl').write_text(''.join(json.dumps(a, sort_keys=True)+'\n' for a in assignments), encoding='utf-8')
assignments

In [ ]:
from combat_id_calibration.hypothesis_generation import build_llm_hypothesis_prompt
explanation_payload = {
    'scenario_id':'LuaHistory_2026-06-23_first_contact',
    'observation': asdict(obs),
    'emitter_aliases': emitter_aliases(obs.emission_sensor_name),
    'probability_assignments': assignments,
    'llm_instruction': 'Explain why the probability layer favored these identities/operators. Do not change probabilities.',
    'llm_prompt': build_llm_hypothesis_prompt(obs, hypotheses, len(hypotheses)),
}
(WORK_DIR / 'first_contact_explanation_payload.json').write_text(json.dumps(explanation_payload, indent=2, sort_keys=True), encoding='utf-8')
explanation_payload